<a href="https://colab.research.google.com/github/hussainzia/python-mini-projects/blob/main/Project9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Project 9: Data Explorer

Loads a CSV file, shows a preview and summary statistics, lets you filter rows by a column value, and draws a line chart of any two columns.

If `sample_data.csv` isn't next to the notebook (for example, on a fresh Colab runtime), it uses a built-in sample dataset, so it always runs.

**Concepts:** pandas DataFrames, filtering, `describe()`, matplotlib charts.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

CSV_PATH = Path("sample_data.csv")

SAMPLE_DATA = {
    "Date": pd.date_range("2024-01-01", periods=10).strftime("%Y-%m-%d"),
    "Category": ["A", "B", "A", "C", "B", "A", "C", "B", "A", "C"],
    "Sales": [100, 200, 150, 300, 250, 180, 320, 275, 190, 295],
    "Profit": [10, 20, 15, 30, 25, 18, 32, 27, 19, 30],
}


def load_data(path: Path = CSV_PATH) -> pd.DataFrame:
    if path.exists():
        print(f"Loaded {path}")
        return pd.read_csv(path)
    print(f"'{path}' not found, so I'm using the built-in sample data.")
    return pd.DataFrame(SAMPLE_DATA)


def choose_column(df: pd.DataFrame, prompt: str, numeric: bool = False) -> str:
    """Ask for a column name (case-insensitive) until a valid one is given."""
    columns = list(df.columns)
    lookup = {c.lower(): c for c in columns}
    while True:
        column = lookup.get(input(prompt).strip().lower())
        if column is None:
            print(f"Please choose one of: {', '.join(columns)}")
        elif numeric and not pd.api.types.is_numeric_dtype(df[column]):
            print(f"'{column}' isn't numeric, so it can't go on the y-axis. Pick another column.")
        else:
            return column


def filter_rows(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Keep only rows where `column` equals a value the user types (Enter keeps everything)."""
    values = ", ".join(map(str, df[column].unique()))
    raw = input(f"Values in {column}: {values}\nFilter value (press Enter to keep all rows): ").strip()
    if not raw:
        return df

    if pd.api.types.is_numeric_dtype(df[column]):
        try:
            mask = df[column] == float(raw)
        except ValueError:
            print("That isn't a number, so I'm keeping all rows.")
            return df
    else:
        mask = df[column].astype(str).str.lower() == raw.lower()

    if not mask.any():
        print("No rows match, so I'm keeping all rows.")
        return df
    return df[mask]


def plot(df: pd.DataFrame, x: str, y: str) -> None:
    data = df.sort_values(x)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(data[x].astype(str), data[y], marker="o", linewidth=2, color="#2563eb")
    ax.set(xlabel=x, ylabel=y, title=f"{y} by {x}")
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def main() -> None:
    df = load_data()
    print("\nData preview:")
    print(df.head())
    print("\nSummary statistics:")
    print(df.describe())

    print(f"\nColumns: {', '.join(df.columns)}")
    filter_column = choose_column(df, "Column to filter by: ")
    filtered = filter_rows(df, filter_column)
    print(f"\nFiltered data ({len(filtered)} rows):")
    print(filtered)

    x = choose_column(df, "Column for the x-axis: ")
    y = choose_column(df, "Column for the y-axis (numeric): ", numeric=True)
    plot(filtered, x, y)


if __name__ == "__main__":
    main()